In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, ndcg_score
from imblearn.over_sampling import SMOTE
import optuna
from optuna.samplers import TPESampler
from sklearn.inspection import permutation_importance
import os

RANDOM_STATE = 42

# Φόρτωση των συνόλων από το 03_models
train_fit = pd.read_pickle("train_fit.pkl")
valid     = pd.read_pickle("valid.pkl")
train     = pd.read_pickle("train.pkl")
test      = pd.read_pickle("test.pkl")
print("train_fit:", train_fit.shape, "| valid:", valid.shape, "| train:", train.shape, "| test:", test.shape)

# Μεταβλητές εισόδου που επιλέχθηκαν στο 03: raw_scaled (25 raw / ενεργητικό + at, prcc_f, csho)
raw_cols = ['act','ap','at','ceq','che','cogs','csho','dlc','dltis','dltt','dp','ib',
            'invt','ivao','ivst','lct','lt','ni','ppegt','pstk','re','rect','sale',
            'sstk','txp','txt','xint','prcc_f']
keep_as_is = ['at', 'prcc_f', 'csho']
scaled_cols = []
for c in raw_cols:
    if c not in keep_as_is:
        scaled_cols.append(c + '_at')
raw_scaled = scaled_cols + keep_as_is
print("Πλήθος raw_scaled:", len(raw_scaled))   # πρέπει 28

# NDCG@1% : ανά έτος, k = 1% των παρατηρήσεων του έτους, μέσος όρος των ετών
def ndcg_1pct(data, prediction):
    scores = []
    for year in sorted(data['fyear'].unique()):
        year_mask = (data['fyear'] == year).values
        k = round(0.01 * year_mask.sum())
        score = ndcg_score([data.loc[year_mask, 'misstate']], [prediction[year_mask]], k=k)
        scores.append(score)
    return np.mean(scores)

train_fit: (54052, 78) | valid: (13114, 78) | train: (67166, 78) | test: (35166, 78)
Πλήθος raw_scaled: 28


c:\Users\Φαίδρα\ΑΣΧΕΤΑ\Desktop\ΜΕΤΑΠΤΥΧΙΑΚΟ\Γ' ΕΞΑΜΗΝΟ\ΠΤΥΧΙΑΚΗ\ptixiaki_python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def confusion_top_k(data, prediction, k=100):
    tp, fp, fn = 0, 0, 0
    for year in sorted(data['fyear'].unique()):
        year_mask = (data['fyear'] == year).values           # True για τις γραμμές αυτού του έτους
        y_year = data.loc[year_mask, 'misstate'].values      # πραγματικές ετικέτες του έτους
        p_year = prediction[year_mask]                       # πιθανότητες του μοντέλου για το έτος

        # np.argsort(-p_year): οι θέσεις των εταιρειών ταξινομημένες από τη μεγαλύτερη
        # στη μικρότερη πιθανότητα. Οι k πρώτες θέσεις είναι αυτές που ελέγχονται.
        order = np.argsort(-p_year)
        checked = np.zeros(len(p_year), dtype=bool)          # αρχικά καμία εταιρεία δεν ελέγχεται
        checked[order[:k]] = True                            # ελέγχονται οι k πρώτες

        tp += (checked & (y_year == 1)).sum()     # ελέγχθηκε και ήταν απάτη
        fp += (checked & (y_year == 0)).sum()     # ελέγχθηκε αλλά δεν ήταν απάτη
        fn += (~checked & (y_year == 1)).sum()    # δεν ελέγχθηκε αλλά ήταν απάτη
    return tp, fp, fn

### **ΑΝΙΣΟΡΡΟΠΙΑ ΚΛΑΣΕΩΝ**

In [3]:
# Βάση: raw_scaled χωρίς χειρισμό ανισορροπίας 
model_base = XGBClassifier(random_state=RANDOM_STATE)
model_base.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_base = model_base.predict_proba(valid[raw_scaled])[:, 1]
auc_base = roc_auc_score(valid['misstate'], prediction_base)
ndcg_base = ndcg_1pct(valid, prediction_base)

# Παραλλαγή 1: scale_pos_weight (βάρος στις απάτες = μη απάτες / απάτες του train_fit)
n_neg = (train_fit['misstate'] == 0).sum()
n_pos = (train_fit['misstate'] == 1).sum()
spw = n_neg / n_pos
print("Μη απάτες:", n_neg, "| Απάτες:", n_pos, "| scale_pos_weight:", round(spw, 1))

model_spw = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=spw)
model_spw.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_spw = model_spw.predict_proba(valid[raw_scaled])[:, 1]
auc_spw = roc_auc_score(valid['misstate'], prediction_spw)
ndcg_spw = ndcg_1pct(valid, prediction_spw)

print("Καμία (βάση):      AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print("scale_pos_weight:  AUC", round(auc_spw, 4),  "| NDCG@1%", round(ndcg_spw, 4))

Μη απάτες: 53808 | Απάτες: 244 | scale_pos_weight: 220.5
Καμία (βάση):      AUC 0.7329 | NDCG@1% 0.1396
scale_pos_weight:  AUC 0.6973 | NDCG@1% 0.0617


##### Παρατηρούμε οτι για scale_pos_weight = 220.5 μειώθηκε και το AUC και το NDCG@1%

### **SCALE_POS_WEIGHT**

In [4]:
# scale_pos_weight: δοκιμή τιμών
spw_values = [5, 10, 25, 50, 100, round(spw, 1)]
spw_results = []

for w in spw_values:
    model_w = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=w)
    model_w.fit(train_fit[raw_scaled], train_fit['misstate'])
    prediction_w = model_w.predict_proba(valid[raw_scaled])[:, 1]
    spw_results.append((w,
                        roc_auc_score(valid['misstate'], prediction_w),
                        ndcg_1pct(valid, prediction_w)))

spw_table = pd.DataFrame(spw_results, columns=['scale_pos_weight', 'AUC', 'NDCG@1%'])
print("Βάση (καμία): AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print(spw_table.round(4))

Βάση (καμία): AUC 0.7329 | NDCG@1% 0.1396
   scale_pos_weight     AUC  NDCG@1%
0               5.0  0.7509   0.1135
1              10.0  0.7288   0.1142
2              25.0  0.7053   0.1033
3              50.0  0.7231   0.0664
4             100.0  0.7269   0.0791
5             220.5  0.6973   0.0617


##### Παρατηρούμε ότι για scale_pos_weight = 5 βελτιώνεται το AUC ενώ μειώνεται λίγο το NDCG@1% ενώ για όλα σε όλες τις υπόλοιπες δοκιμές μειώνονται και τα δύο σε σχέση με το αρχικό. Συνεπώς κρατλαμε το scale_pos_weight = 5 καθώς έχουμε ως βασικό μετρο το AUC και δευτεύον το NDCG@1%.

### **UNDERSAMPLING**

In [5]:
#  Random Undersampling (μόνο στο train_fit): κρατάμε όλες τις απάτες
# και τυχαίο δείγμα μη απατών, ώστε απάτες : μη απάτες = 1:1, 1:4, 1:10
frauds = train_fit[train_fit['misstate'] == 1]
non_frauds = train_fit[train_fit['misstate'] == 0]

rus_ratios = [1, 4, 10]          # πόσες μη απάτες για κάθε 1 απάτη
rus_results = []

for k in rus_ratios:
    non_sample = non_frauds.sample(n=len(frauds) * k, random_state=RANDOM_STATE)
    train_rus = pd.concat([frauds, non_sample])

    model_r = XGBClassifier(random_state=RANDOM_STATE)
    model_r.fit(train_rus[raw_scaled], train_rus['misstate'])
    prediction_r = model_r.predict_proba(valid[raw_scaled])[:, 1]

    rus_results.append(('1:' + str(k), len(train_rus),
                        roc_auc_score(valid['misstate'], prediction_r),
                        ndcg_1pct(valid, prediction_r)))

rus_table = pd.DataFrame(rus_results, columns=['Αναλογία', 'Γραμμές train', 'AUC', 'NDCG@1%'])
print("Βάση (καμία): AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print(rus_table.round(4))

Βάση (καμία): AUC 0.7329 | NDCG@1% 0.1396
  Αναλογία  Γραμμές train     AUC  NDCG@1%
0      1:1            488  0.7177   0.1331
1      1:4           1220  0.7386   0.1078
2     1:10           2684  0.7177   0.1162


 Παρατηρούμε ότι για random undersampling με αναλογια 1:4 παρουσίασε μεγαλύτερο AUC κατά 0,0057 ενώ μειώθηκε το NDCG@1%.Αυτό σημαίνει πως βελτιώθηκε η συνολική κατάταξη αλλα΄όχι η κορυφή της λίστας. Ωστόσο, το AUC 0.7386 αυτης της μεθόδου ήταν μικρότερο από αυτο του scale_pos_weight = 5 που ήταν ίσο με 0,7509

In [6]:
# Έλεγχος σταθερότητας: σκέτο RUS 1:4 με 5 διαφορετικές τυχαίες επιλογές
seeds = [1, 21, 42]
rus_seed_results = []

for s in seeds:
    non_sample = non_frauds.sample(n=len(frauds) * 4, random_state=s)
    train_r = pd.concat([frauds, non_sample])

    model_r = XGBClassifier(random_state=RANDOM_STATE)
    model_r.fit(train_r[raw_scaled], train_r['misstate'])
    prediction_r = model_r.predict_proba(valid[raw_scaled])[:, 1]
    rus_seed_results.append((s, roc_auc_score(valid['misstate'], prediction_r),
                             ndcg_1pct(valid, prediction_r)))

rus_seed_table = pd.DataFrame(rus_seed_results, columns=['random_state', 'AUC', 'NDCG@1%'])
print(rus_seed_table.round(4))
print("Μέσος όρος: AUC", round(rus_seed_table['AUC'].mean(), 4), "| NDCG@1%", round(rus_seed_table['NDCG@1%'].mean(), 4))
print("Ελάχιστο - μέγιστο AUC:", round(rus_seed_table['AUC'].min(), 4), "-", round(rus_seed_table['AUC'].max(), 4))

   random_state     AUC  NDCG@1%
0             1  0.7461   0.1137
1            21  0.7431   0.0963
2            42  0.7386   0.1078
Μέσος όρος: AUC 0.7426 | NDCG@1% 0.1059
Ελάχιστο - μέγιστο AUC: 0.7386 - 0.7461


Παρατηρούμε ότι γενικά το AUC έχει μια σταθερότητα με μέσο όρο 0,7447 ενώ το NDCG παρουσιάζει ένα λίγο μεγαλύτερο έυρος με μέσο όρο 0,0989

### **SMOTE**

In [7]:
# Το SMOTE δεν δέχεται κενές τιμές: συμπλήρωση με τη διάμεσο του train_fit.
# Οι ίδιες διάμεσοι μπαίνουν και στο validation (υπολογίζονται μόνο από το train_fit -> καμία διαρροή).
medians = train_fit[raw_scaled].median()
X_fit = train_fit[raw_scaled].fillna(medians)
y_fit = train_fit['misstate']
X_valid_filled = valid[raw_scaled].fillna(medians)

smote_results = []

# Έλεγχος: μόνο συμπλήρωση, χωρίς SMOTE (για να ξεχωρίσουμε την επίδραση της συμπλήρωσης)
model_imp = XGBClassifier(random_state=RANDOM_STATE)
model_imp.fit(X_fit, y_fit)
prediction_imp = model_imp.predict_proba(X_valid_filled)[:, 1]
smote_results.append(('Μόνο συμπλήρωση', int(y_fit.sum()), len(y_fit),
                      roc_auc_score(valid['misstate'], prediction_imp),
                      ndcg_1pct(valid, prediction_imp)))

# SMOTE (k_neighbors = 5, όπως Chawla et al., 2002) - μόνο στο train_fit
smote_ratios = [10, 4, 1]        # μη απάτες για κάθε 1 απάτη μετά το SMOTE
for k in smote_ratios:
    smote = SMOTE(sampling_strategy=1 / k, random_state=RANDOM_STATE)
    X_sm, y_sm = smote.fit_resample(X_fit, y_fit)

    model_s = XGBClassifier(random_state=RANDOM_STATE)
    model_s.fit(X_sm, y_sm)
    prediction_s = model_s.predict_proba(X_valid_filled)[:, 1]

    smote_results.append(('SMOTE 1:' + str(k), int(y_sm.sum()), len(y_sm),
                          roc_auc_score(valid['misstate'], prediction_s),
                          ndcg_1pct(valid, prediction_s)))

smote_table = pd.DataFrame(smote_results,
                           columns=['Παραλλαγή', 'Απάτες στο train', 'Γραμμές train', 'AUC', 'NDCG@1%'])
print("Βάση (καμία): AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print(smote_table.round(4))

Βάση (καμία): AUC 0.7329 | NDCG@1% 0.1396
         Παραλλαγή  Απάτες στο train  Γραμμές train     AUC  NDCG@1%
0  Μόνο συμπλήρωση               244          54052  0.7322   0.1099
1       SMOTE 1:10              5380          59188  0.6846   0.0379
2        SMOTE 1:4             13452          67260  0.6810   0.0349
3        SMOTE 1:1             53808         107616  0.6562   0.0380


 Η μέθοδος SMOTE αρχικά δεν μπορεί να λειρουργήσει με κενές τιμές οπότε συμπληρώθηκε η διάμεσος για τις ελλειπούσες τιμες με αποτέλεσμα το AUC να μειωθεί από 0,7329 σε 0,7322. Επίσης, δεν βελτίωσε με καμία αναλογία το αρχικό μοντέλο, αντίθετα μειώθηκε αρκετα.

### **ΣΥΝΔΥΑΣΜΟΣ UNDERSAMPLING + SMOTE**

In [10]:
# Συνδυασμός: Random Undersampling και μετά SMOTE (Chawla et al., 2002) - ΜΟΝΟ στο train_fit
combos = [(10, 4), (10, 1), (4, 1)]   # (μη απάτες ανά απάτη μετά το undersampling, μετά το SMOTE)
combo_results = []

for k_rus, k_smote in combos:
    # 1) Undersampling: όλες οι απάτες + τυχαίο δείγμα μη απατών
    non_sample = non_frauds.sample(n=len(frauds) * k_rus, random_state=RANDOM_STATE)
    train_c = pd.concat([frauds, non_sample])

    # 2) Συμπλήρωση κενών (με τις ίδιες διαμέσους του train_fit) και SMOTE
    X_c = train_c[raw_scaled].fillna(medians)
    y_c = train_c['misstate']
    X_cs, y_cs = SMOTE(sampling_strategy=1 / k_smote, random_state=RANDOM_STATE).fit_resample(X_c, y_c)

    # 3) Εκπαίδευση και αξιολόγηση στο κανονικό validation
    model_c = XGBClassifier(random_state=RANDOM_STATE)
    model_c.fit(X_cs, y_cs)
    prediction_c = model_c.predict_proba(X_valid_filled)[:, 1]

    combo_results.append(('RUS 1:' + str(k_rus) + ' → SMOTE 1:' + str(k_smote),
                          int(y_cs.sum()), len(y_cs),
                          roc_auc_score(valid['misstate'], prediction_c),
                          ndcg_1pct(valid, prediction_c)))

combo_table = pd.DataFrame(combo_results,
                           columns=['Συνδυασμός', 'Απάτες στο train', 'Γραμμές train', 'AUC', 'NDCG@1%'])
print("Βάση (καμία): AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print(combo_table.round(4))

Βάση (καμία): AUC 0.7329 | NDCG@1% 0.1396
             Συνδυασμός  Απάτες στο train  Γραμμές train     AUC  NDCG@1%
0  RUS 1:10 → SMOTE 1:4               610           3050  0.7184   0.0971
1  RUS 1:10 → SMOTE 1:1              2440           4880  0.6993   0.0591
2   RUS 1:4 → SMOTE 1:1               976           1952  0.7474   0.1500


In [11]:
# Έλεγχος σταθερότητας: RUS 1:4 → SMOTE 1:1 με 5 διαφορετικές τυχαίες επιλογές
seeds = [1, 21, 42]
seed_results = []

for s in seeds:
    non_sample = non_frauds.sample(n=len(frauds) * 4, random_state=s)
    train_c = pd.concat([frauds, non_sample])
    X_c = train_c[raw_scaled].fillna(medians)
    y_c = train_c['misstate']
    X_cs, y_cs = SMOTE(sampling_strategy=1.0, random_state=s).fit_resample(X_c, y_c)

    model_c = XGBClassifier(random_state=RANDOM_STATE)
    model_c.fit(X_cs, y_cs)
    prediction_c = model_c.predict_proba(X_valid_filled)[:, 1]
    seed_results.append((s, roc_auc_score(valid['misstate'], prediction_c),
                         ndcg_1pct(valid, prediction_c)))

seed_table = pd.DataFrame(seed_results, columns=['random_state', 'AUC', 'NDCG@1%'])
print(seed_table.round(4))
print("Μέσος όρος: AUC", round(seed_table['AUC'].mean(), 4), "| NDCG@1%", round(seed_table['NDCG@1%'].mean(), 4))
print("Ελάχιστο - μέγιστο AUC:", round(seed_table['AUC'].min(), 4), "-", round(seed_table['AUC'].max(), 4))

   random_state     AUC  NDCG@1%
0             1  0.7469   0.0683
1            21  0.7315   0.0902
2            42  0.7474   0.1500
Μέσος όρος: AUC 0.7419 | NDCG@1% 0.1028
Ελάχιστο - μέγιστο AUC: 0.7315 - 0.7474


Παρατηρούμε ότι ο μέσος όρος του AUC είναι σχεδόν ίδιος με τον αρχικό και γενικά έχει σχετικά μικρό εύρος (0,7315 - 0,7630) σε αντίθεση με το NDCG@1% το οποίο είναι αρκετά ασταθές και ήταν τυχαίο πριν το 0,15. Ο μέσος όρος είναι 0,0939 που βρίσκεται κάτω από το βασικό μοντέλο (model_4_scaled_raw με NDCG@1% 0,1396 ) και κάτω από το scale_pos_weight = 5 με NDCG@1% = 0,1135. Άρα το αποτέλεσμα του συνδυασμού στο NDCG δεν ήταν πραγματικό αλλά τυχαίο. 


In [17]:
# Αντίστροφη σειρά: πρώτα SMOTE (σε όλο το train_fit), μετά Random Undersampling
reverse_combos = [(10, 4), (10, 1), (4, 1)]   # (αναλογία μετά το SMOTE, αναλογία μετά το undersampling)
reverse_results = []

for k_smote, k_rus in reverse_combos:
    # 1) SMOTE στο πλήρες train_fit (με τα ήδη συμπληρωμένα X_fit, y_fit)
    X_s, y_s = SMOTE(sampling_strategy=1 / k_smote, random_state=RANDOM_STATE).fit_resample(X_fit, y_fit)

    # Ενώνουμε X και y σε ένα πίνακα για να μπορούμε να χωρίσουμε απάτες / μη απάτες
    data_s = X_s.copy()
    data_s['misstate'] = y_s.values
    frauds_s = data_s[data_s['misstate'] == 1]
    non_frauds_s = data_s[data_s['misstate'] == 0]

    # 2) Undersampling: όλες οι απάτες (πραγματικές + τεχνητές) + τυχαίο δείγμα μη απατών
    non_sample_s = non_frauds_s.sample(n=len(frauds_s) * k_rus, random_state=RANDOM_STATE)
    train_sr = pd.concat([frauds_s, non_sample_s])

    # 3) Εκπαίδευση και αξιολόγηση στο κανονικό validation
    model_sr = XGBClassifier(random_state=RANDOM_STATE)
    model_sr.fit(train_sr[raw_scaled], train_sr['misstate'])
    prediction_sr = model_sr.predict_proba(X_valid_filled)[:, 1]

    reverse_results.append(('SMOTE 1:' + str(k_smote) + ' → RUS 1:' + str(k_rus),
                            int(train_sr['misstate'].sum()), len(train_sr),
                            roc_auc_score(valid['misstate'], prediction_sr),
                            ndcg_1pct(valid, prediction_sr)))

reverse_table = pd.DataFrame(reverse_results,
                             columns=['Συνδυασμός', 'Απάτες στο train', 'Γραμμές train', 'AUC', 'NDCG@1%'])
print("Βάση (καμία): AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print(reverse_table.round(4))

Βάση (καμία): AUC 0.7329 | NDCG@1% 0.1396
             Συνδυασμός  Απάτες στο train  Γραμμές train     AUC  NDCG@1%
0  SMOTE 1:10 → RUS 1:4              5380          26900  0.6838   0.0425
1  SMOTE 1:10 → RUS 1:1              5380          10760  0.7020   0.0403
2   SMOTE 1:4 → RUS 1:1             13452          26904  0.6873   0.0429


Το AUC με τον συνδυασμό SMOTE και μετά undersampling είναι αρκετά μικρότερο των προηγούμενων παραλλαγών όπως και το NDCG@1%. 

Συνεπώς το **scale_pos_weight = 5** είναι *νικητής* με AUC  0.7509 και NDCG@1% = 0.1135. Σημειώνεται επίσης πως δεν εχει τυχαιότητα το μοντέλο αυτο και χρησιμοποιεί όλα τα δεδομένα.
H δεύτερη καλύτερη επιλογή είναι ο RUS 1:4 AUC 0.7426 και NDCG@1% = 0,1059

## **OPTUNA**

### **Εκδοχή Α**: scale_pos_weight σταθερό = 5 (νικητής), το Optuna ψάχνει τις 6 ρυθμίσεις

In [19]:
optuna.logging.set_verbosity(optuna.logging.WARNING)   # να μην τυπώνει γραμμή για κάθε δοκιμή

def objective_spw_fixed(trial):
    n_estimators     = trial.suggest_int('n_estimators', 100, 2000)
    learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth        = trial.suggest_int('max_depth', 3, 12)
    min_child_weight = trial.suggest_float('min_child_weight', 1, 7)
    subsample        = trial.suggest_float('subsample', 0.4, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0)

    model = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=5,
                          n_estimators=n_estimators, learning_rate=learning_rate,
                          max_depth=max_depth, min_child_weight=min_child_weight,
                          subsample=subsample, colsample_bytree=colsample_bytree)
    model.fit(train_fit[raw_scaled], train_fit['misstate'])
    prediction = model.predict_proba(valid[raw_scaled])[:, 1]
    return roc_auc_score(valid['misstate'], prediction)

study_A_spw = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_A_spw.optimize(objective_spw_fixed, n_trials=50)

print("Καλύτερο AUC στο validation:", round(study_A_spw.best_value, 4))
print("Καλύτερες ρυθμίσεις:")
print(study_A_spw.best_params)

Καλύτερο AUC στο validation: 0.7905
Καλύτερες ρυθμίσεις:
{'n_estimators': 496, 'learning_rate': 0.015258454459097485, 'max_depth': 4, 'min_child_weight': 2.599630437518745, 'subsample': 0.5219865275854596, 'colsample_bytree': 0.4395595092783027}


In [20]:
# Εκδοχή Α: ξαναεκπαίδευση με τις καλύτερες ρυθμίσεις -> έλεγχος AUC και υπολογισμός NDCG@1%
best_A_spw = study_A_spw.best_params

model_A_spw = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=5,
                            n_estimators=best_A_spw['n_estimators'],
                            learning_rate=best_A_spw['learning_rate'],
                            max_depth=best_A_spw['max_depth'],
                            min_child_weight=best_A_spw['min_child_weight'],
                            subsample=best_A_spw['subsample'],
                            colsample_bytree=best_A_spw['colsample_bytree'])
model_A_spw.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_A_spw = model_A_spw.predict_proba(valid[raw_scaled])[:, 1]

auc_A_spw = roc_auc_score(valid['misstate'], prediction_A_spw)
ndcg_A_spw = ndcg_1pct(valid, prediction_A_spw)
print("scale_pos_weight=5, προεπιλογές:  AUC 0.7509 | NDCG@1% 0.1135")
print("scale_pos_weight=5, Optuna (Α):   AUC", round(auc_A_spw, 4), "| NDCG@1%", round(ndcg_A_spw, 4))

scale_pos_weight=5, προεπιλογές:  AUC 0.7509 | NDCG@1% 0.1135
scale_pos_weight=5, Optuna (Α):   AUC 0.7905 | NDCG@1% 0.1579


Με τη ρύθμιση των υπερπαραμέτρων (Optuna, 50 δοκιμές), το AUC αυξήθηκε από 0,7509 σε 0,7905 και το NDCG@1% από 0,1135 σε 0,1579. Οι ρυθμίσεις που επιλέχθηκαν (μικρό learning_rate, ρηχά δέντρα, χρήση μέρους των γραμμών και των μεταβλητών σε κάθε δέντρο) οδηγούν σε απλούστερο μοντέλο, κάτι που ταιριάζει με τον μικρό αριθμό απατών του δείγματος.

### **Εκδοχή Β** :και το scale_pos_weight μέσα στο Optuna

In [21]:
# Εκδοχή Β: το Optuna ψάχνει και το scale_pos_weight μαζί με τις 6 ρυθμίσεις
def objective_spw_tuned(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1, 220.5, log=True)
    n_estimators     = trial.suggest_int('n_estimators', 100, 2000)
    learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth        = trial.suggest_int('max_depth', 3, 12)
    min_child_weight = trial.suggest_float('min_child_weight', 1, 7)
    subsample        = trial.suggest_float('subsample', 0.4, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0)

    model = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight,
                          n_estimators=n_estimators, learning_rate=learning_rate,
                          max_depth=max_depth, min_child_weight=min_child_weight,
                          subsample=subsample, colsample_bytree=colsample_bytree)
    model.fit(train_fit[raw_scaled], train_fit['misstate'])
    prediction = model.predict_proba(valid[raw_scaled])[:, 1]
    return roc_auc_score(valid['misstate'], prediction)

study_B_spw = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_B_spw.optimize(objective_spw_tuned, n_trials=50)
best_B_spw = study_B_spw.best_params
print("Καλύτερες ρυθμίσεις:", best_B_spw)

# Ξαναεκπαίδευση με τις καλύτερες ρυθμίσεις -> AUC και NDCG@1%
model_B_spw = XGBClassifier(random_state=RANDOM_STATE,
                            scale_pos_weight=best_B_spw['scale_pos_weight'],
                            n_estimators=best_B_spw['n_estimators'],
                            learning_rate=best_B_spw['learning_rate'],
                            max_depth=best_B_spw['max_depth'],
                            min_child_weight=best_B_spw['min_child_weight'],
                            subsample=best_B_spw['subsample'],
                            colsample_bytree=best_B_spw['colsample_bytree'])
model_B_spw.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_B_spw = model_B_spw.predict_proba(valid[raw_scaled])[:, 1]
auc_B_spw = roc_auc_score(valid['misstate'], prediction_B_spw)
ndcg_B_spw = ndcg_1pct(valid, prediction_B_spw)

print("scale_pos_weight=5, Optuna (Α):          AUC", round(auc_A_spw, 4), "| NDCG@1%", round(ndcg_A_spw, 4))
print("scale_pos_weight μέσα στο Optuna (Β):    AUC", round(auc_B_spw, 4), "| NDCG@1%", round(ndcg_B_spw, 4))

Καλύτερες ρυθμίσεις: {'scale_pos_weight': 1.2883333782360646, 'n_estimators': 1057, 'learning_rate': 0.011875020655065738, 'max_depth': 4, 'min_child_weight': 3.852260948765374, 'subsample': 0.6771006305632494, 'colsample_bytree': 0.8851320845005959}
scale_pos_weight=5, Optuna (Α):          AUC 0.7905 | NDCG@1% 0.1579
scale_pos_weight μέσα στο Optuna (Β):    AUC 0.7942 | NDCG@1% 0.142


### **Eκδοχή Α** : RUS 1:4 με σταθερή αναλογία

In [13]:
# RUS 1:4, εκδοχή Α: σταθερή αναλογία 1:4, μέσος όρος 3 seeds σε κάθε δοκιμή
optuna.logging.set_verbosity(optuna.logging.WARNING)   # να μην τυπώνει γραμμή για κάθε δοκιμή
rus_seeds = [1, 21, 42]          # ίδια seeds με τον συνδυασμό

def make_rus_train(seed, k_rus):
    # όλες οι απάτες + τυχαίο δείγμα μη απατών (k_rus μη απάτες ανά απάτη)
    non_sample = non_frauds.sample(n=len(frauds) * k_rus, random_state=seed)
    train_r = pd.concat([frauds, non_sample])
    return train_r[raw_scaled], train_r['misstate']

# Τα 3 σύνολα εκπαίδευσης φτιάχνονται μία φορά (η αναλογία δεν αλλάζει σε αυτή την εκδοχή)
rus_sets_A = []
for s in rus_seeds:
    rus_sets_A.append(make_rus_train(s, 4))

def objective_rus_fixed(trial):
    n_estimators     = trial.suggest_int('n_estimators', 100, 2000)
    learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth        = trial.suggest_int('max_depth', 3, 12)
    min_child_weight = trial.suggest_float('min_child_weight', 1, 7)
    subsample        = trial.suggest_float('subsample', 0.4, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0)

    aucs = []
    for X_r, y_r in rus_sets_A:
        model = XGBClassifier(random_state=RANDOM_STATE,
                              n_estimators=n_estimators, learning_rate=learning_rate,
                              max_depth=max_depth, min_child_weight=min_child_weight,
                              subsample=subsample, colsample_bytree=colsample_bytree)
        model.fit(X_r, y_r)
        prediction = model.predict_proba(valid[raw_scaled])[:, 1]
        aucs.append(roc_auc_score(valid['misstate'], prediction))
    return np.mean(aucs)

study_A_rus = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_A_rus.optimize(objective_rus_fixed, n_trials=50)
best_A_rus = study_A_rus.best_params
print("Καλύτερες ρυθμίσεις:", best_A_rus)

# Σύγκριση στα ίδια 3 σύνολα: προεπιλογές έναντι καλύτερων ρυθμίσεων (μέσος όρος AUC και NDCG@1%)
default_rus = []
tuned_rus = []
for X_r, y_r in rus_sets_A:
    model_d = XGBClassifier(random_state=RANDOM_STATE)
    model_d.fit(X_r, y_r)
    pred_d = model_d.predict_proba(valid[raw_scaled])[:, 1]
    default_rus.append((roc_auc_score(valid['misstate'], pred_d), ndcg_1pct(valid, pred_d)))

    model_t = XGBClassifier(random_state=RANDOM_STATE,
                            n_estimators=best_A_rus['n_estimators'],
                            learning_rate=best_A_rus['learning_rate'],
                            max_depth=best_A_rus['max_depth'],
                            min_child_weight=best_A_rus['min_child_weight'],
                            subsample=best_A_rus['subsample'],
                            colsample_bytree=best_A_rus['colsample_bytree'])
    model_t.fit(X_r, y_r)
    pred_t = model_t.predict_proba(valid[raw_scaled])[:, 1]
    tuned_rus.append((roc_auc_score(valid['misstate'], pred_t), ndcg_1pct(valid, pred_t)))

default_rus = pd.DataFrame(default_rus, columns=['AUC', 'NDCG@1%'])
tuned_rus = pd.DataFrame(tuned_rus, columns=['AUC', 'NDCG@1%'])
print("RUS 1:4, προεπιλογές (μ.ό. 3 seeds):  AUC", round(default_rus['AUC'].mean(), 4), "| NDCG@1%", round(default_rus['NDCG@1%'].mean(), 4))
print("RUS 1:4, Optuna Α (μ.ό. 3 seeds):     AUC", round(tuned_rus['AUC'].mean(), 4), "| NDCG@1%", round(tuned_rus['NDCG@1%'].mean(), 4))

Καλύτερες ρυθμίσεις: {'n_estimators': 1244, 'learning_rate': 0.010662922953194398, 'max_depth': 3, 'min_child_weight': 5.1946591892879574, 'subsample': 0.6110672891443766, 'colsample_bytree': 0.7643179227699156}
RUS 1:4, προεπιλογές (μ.ό. 3 seeds):  AUC 0.7426 | NDCG@1% 0.1059
RUS 1:4, Optuna Α (μ.ό. 3 seeds):     AUC 0.7757 | NDCG@1% 0.144


### **Εκδοχή Β** : το Optuna ψάχνει αναλογίες undersampling 

In [14]:
optuna.logging.set_verbosity(optuna.logging.WARNING)   # να μην τυπώνει γραμμή για κάθε δοκιμή

# RUS, εκδοχή Β: το Optuna ψάχνει και την αναλογία undersampling μαζί με τις 6 ρυθμίσεις
def objective_rus_tuned(trial):
    k_rus            = trial.suggest_int('k_rus', 1, 10)     # μη απάτες ανά απάτη μετά το undersampling
    n_estimators     = trial.suggest_int('n_estimators', 100, 2000)
    learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth        = trial.suggest_int('max_depth', 3, 12)
    min_child_weight = trial.suggest_float('min_child_weight', 1, 7)
    subsample        = trial.suggest_float('subsample', 0.4, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0)

    aucs = []
    for s in rus_seeds:
        X_r, y_r = make_rus_train(s, k_rus)    # τα σύνολα φτιάχνονται εδώ, γιατί η αναλογία αλλάζει
        model = XGBClassifier(random_state=RANDOM_STATE,
                              n_estimators=n_estimators, learning_rate=learning_rate,
                              max_depth=max_depth, min_child_weight=min_child_weight,
                              subsample=subsample, colsample_bytree=colsample_bytree)
        model.fit(X_r, y_r)
        prediction = model.predict_proba(valid[raw_scaled])[:, 1]
        aucs.append(roc_auc_score(valid['misstate'], prediction))
    return np.mean(aucs)

study_B_rus = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_B_rus.optimize(objective_rus_tuned, n_trials=50)
best_B_rus = study_B_rus.best_params
print("Καλύτερες ρυθμίσεις:", best_B_rus)

# Ξαναεκπαίδευση με τις καλύτερες ρυθμίσεις στα 3 seeds -> μέσος όρος AUC και NDCG@1%
scores_B_rus = []
for s in rus_seeds:
    X_r, y_r = make_rus_train(s, best_B_rus['k_rus'])
    model_t = XGBClassifier(random_state=RANDOM_STATE,
                            n_estimators=best_B_rus['n_estimators'],
                            learning_rate=best_B_rus['learning_rate'],
                            max_depth=best_B_rus['max_depth'],
                            min_child_weight=best_B_rus['min_child_weight'],
                            subsample=best_B_rus['subsample'],
                            colsample_bytree=best_B_rus['colsample_bytree'])
    model_t.fit(X_r, y_r)
    pred_t = model_t.predict_proba(valid[raw_scaled])[:, 1]
    scores_B_rus.append((roc_auc_score(valid['misstate'], pred_t), ndcg_1pct(valid, pred_t)))

scores_B_rus = pd.DataFrame(scores_B_rus, columns=['AUC', 'NDCG@1%'])
print("RUS, Optuna Α (μ.ό. 3 seeds):  AUC", round(tuned_rus['AUC'].mean(), 4), "| NDCG@1%", round(tuned_rus['NDCG@1%'].mean(), 4))
print("RUS, Optuna Β (μ.ό. 3 seeds):  AUC", round(scores_B_rus['AUC'].mean(), 4), "| NDCG@1%", round(scores_B_rus['NDCG@1%'].mean(), 4))

Καλύτερες ρυθμίσεις: {'k_rus': 7, 'n_estimators': 180, 'learning_rate': 0.019331489777152353, 'max_depth': 11, 'min_child_weight': 5.181324133821986, 'subsample': 0.6675996187691271, 'colsample_bytree': 0.5721646375496795}
RUS, Optuna Α (μ.ό. 3 seeds):  AUC 0.7757 | NDCG@1% 0.144
RUS, Optuna Β (μ.ό. 3 seeds):  AUC 0.784 | NDCG@1% 0.1859


In [ ]:
# Ρυθμίσεις Optuna: αν υπάρχει ήδη το αρχείο φορτώνονται, αλλιώς αποθηκεύονται από τη μνήμη
file_params = "optuna_best_params.pkl"

if os.path.exists(file_params):
    best_params = pd.read_pickle(file_params)
    print("Οι ρυθμίσεις φορτώθηκαν από το αρχείο")
else:
    best_params = {'A_spw': best_A_spw, 'B_spw': best_B_spw,
                   'A_rus': best_A_rus, 'B_rus': best_B_rus}
    pd.to_pickle(best_params, file_params)
    print("Οι ρυθμίσεις αποθηκεύτηκαν στο αρχείο")

best_A_spw = best_params['A_spw']
best_B_spw = best_params['B_spw']
best_A_rus = best_params['A_rus']
best_B_rus = best_params['B_rus']
for name, params in best_params.items():
    print(name, params)

Οι ρυθμίσεις φορτώθηκαν από το αρχείο
A_spw {'n_estimators': 496, 'learning_rate': 0.015258454459097485, 'max_depth': 4, 'min_child_weight': 2.599630437518745, 'subsample': 0.5219865275854596, 'colsample_bytree': 0.4395595092783027}
B_spw {'scale_pos_weight': 1.2883333782360646, 'n_estimators': 1057, 'learning_rate': 0.011875020655065738, 'max_depth': 4, 'min_child_weight': 3.852260948765374, 'subsample': 0.6771006305632494, 'colsample_bytree': 0.8851320845005959}
A_rus {'n_estimators': 1244, 'learning_rate': 0.010662922953194398, 'max_depth': 3, 'min_child_weight': 5.1946591892879574, 'subsample': 0.6110672891443766, 'colsample_bytree': 0.7643179227699156}
B_rus {'k_rus': 7, 'n_estimators': 180, 'learning_rate': 0.019331489777152353, 'max_depth': 11, 'min_child_weight': 5.181324133821986, 'subsample': 0.6675996187691271, 'colsample_bytree': 0.5721646375496795}


In [ ]:
# Συγκεντρωτικός πίνακας 04 (μέρος 1): μοντέλα χωρίς τυχαία δειγματοληψία

# scale_pos_weight = 5: ξαναεκπαίδευση (η πρόβλεψή του δεν κρατήθηκε μέσα στο loop)
model_spw5 = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=5)
model_spw5.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_spw5 = model_spw5.predict_proba(valid[raw_scaled])[:, 1]

# SMOTE 1:10: ξαναεκπαίδευση (η πρόβλεψή του δεν κρατήθηκε μέσα στο loop)
X_sm10, y_sm10 = SMOTE(sampling_strategy=1 / 10, random_state=RANDOM_STATE).fit_resample(X_fit, y_fit)
model_smote10 = XGBClassifier(random_state=RANDOM_STATE)
model_smote10.fit(X_sm10, y_sm10)
prediction_smote10 = model_smote10.predict_proba(X_valid_filled)[:, 1]

# scale_pos_weight, Optuna Α: ξαναεκπαίδευση από τις αποθηκευμένες ρυθμίσεις
model_A_spw = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=5,
                            n_estimators=best_A_spw['n_estimators'],
                            learning_rate=best_A_spw['learning_rate'],
                            max_depth=best_A_spw['max_depth'],
                            min_child_weight=best_A_spw['min_child_weight'],
                            subsample=best_A_spw['subsample'],
                            colsample_bytree=best_A_spw['colsample_bytree'])
model_A_spw.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_A_spw = model_A_spw.predict_proba(valid[raw_scaled])[:, 1]

# scale_pos_weight, Optuna Β: ξαναεκπαίδευση από τις αποθηκευμένες ρυθμίσεις
model_B_spw = XGBClassifier(random_state=RANDOM_STATE,
                            scale_pos_weight=best_B_spw['scale_pos_weight'],
                            n_estimators=best_B_spw['n_estimators'],
                            learning_rate=best_B_spw['learning_rate'],
                            max_depth=best_B_spw['max_depth'],
                            min_child_weight=best_B_spw['min_child_weight'],
                            subsample=best_B_spw['subsample'],
                            colsample_bytree=best_B_spw['colsample_bytree'])
model_B_spw.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_B_spw = model_B_spw.predict_proba(valid[raw_scaled])[:, 1]


models_04_single = [
    ('Βάση (καμία τεχνική)',        prediction_base),
    ('scale_pos_weight = 5',        prediction_spw5),
    ('SMOTE 1:10',                  prediction_smote10),
    ('scale_pos_weight, Optuna Α',  prediction_A_spw),
    ('scale_pos_weight, Optuna Β',  prediction_B_spw),
]

### **FALSE NEGATIVE - FALSE POSITIVE**

In [12]:
# Συγκεντρωτικός πίνακας 04 στο validation: AUC, NDCG@1% και policy (top 100 ανά έτος)
# Όλα τα μοντέλα ξαναεκπαιδεύονται εδώ από τις ρυθμίσεις τους (χωρίς να τρέξει το Optuna)

# Οι προεπιλογές του XGBoost γραμμένες ρητά, ώστε όλα τα μοντέλα να εκπαιδεύονται με τον ίδιο τρόπο
default_params = {'n_estimators': 100, 'learning_rate': 0.3, 'max_depth': 6,
                  'min_child_weight': 1, 'subsample': 1, 'colsample_bytree': 1}

def fit_predict(X, y, params, spw=1):
    # Εκπαίδευση XGBoost με τις δοσμένες ρυθμίσεις -> πιθανότητες απάτης στο validation
    # (spw = 1 σημαίνει καθόλου βάρος, δηλαδή η προεπιλογή)
    model = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=spw,
                          n_estimators=params['n_estimators'],
                          learning_rate=params['learning_rate'],
                          max_depth=params['max_depth'],
                          min_child_weight=params['min_child_weight'],
                          subsample=params['subsample'],
                          colsample_bytree=params['colsample_bytree'])
    model.fit(X, y)
    return model.predict_proba(valid[raw_scaled])[:, 1]

rus_seeds = [1, 21, 42]

def rus_predictions(k_rus, params):
    # RUS: μία πρόβλεψη για κάθε seed (όλες οι απάτες + τυχαίο δείγμα μη απατών)
    preds = []
    for s in rus_seeds:
        non_sample = non_frauds.sample(n=len(frauds) * k_rus, random_state=s)
        train_r = pd.concat([frauds, non_sample])
        preds.append(fit_predict(train_r[raw_scaled], train_r['misstate'], params))
    return preds

X_all = train_fit[raw_scaled]
y_all = train_fit['misstate']

# Για κάθε μοντέλο: (όνομα, λίστα προβλέψεων). 1 πρόβλεψη για τα σταθερά, 3 για το RUS (μία ανά seed)
models_04 = [
    ('Βάση (καμία τεχνική)',        [fit_predict(X_all, y_all, default_params)]),
    ('scale_pos_weight = 5',        [fit_predict(X_all, y_all, default_params, spw=5)]),
    ('scale_pos_weight, Optuna Α',  [fit_predict(X_all, y_all, best_A_spw, spw=5)]),
    ('scale_pos_weight, Optuna Β',  [fit_predict(X_all, y_all, best_B_spw, spw=best_B_spw['scale_pos_weight'])]),
    ('RUS 1:4 (μ.ό. 3 seeds)',      rus_predictions(4, default_params)),
    ('RUS 1:4, Optuna Α (μ.ό. 3 seeds)', rus_predictions(4, best_A_rus)),
    ('RUS, Optuna Β (μ.ό. 3 seeds)',     rus_predictions(best_B_rus['k_rus'], best_B_rus)),
]

rows_04 = []
for name, preds in models_04:
    scores = []
    for pred in preds:
        tp, fp, fn = confusion_top_k(valid, pred, k=100)
        scores.append((roc_auc_score(valid['misstate'], pred), ndcg_1pct(valid, pred), tp, fp, fn))
    m = pd.DataFrame(scores, columns=['AUC', 'NDCG', 'TP', 'FP', 'FN']).mean()   # μ.ό. (για 1 πρόβλεψη = η ίδια)
    rows_04.append((name, m['AUC'], m['NDCG'], m['TP'], m['FP'], m['FN'],
                    m['FP'] / m['TP'],                  # άδικες έρευνες ανά απάτη που βρέθηκε
                    m['TP'] / (m['TP'] + m['FN']),      # sensitivity :  από όλες τις απάτες, πόσες βρέθηκαν
                    m['TP'] / (m['TP'] + m['FP'])))     # precision : από όσες ερευνήθηκαν, πόσες ήταν απάτη

results_04 = pd.DataFrame(rows_04, columns=['Μοντέλο', 'AUC', 'NDCG@1%', 'TP', 'FP', 'FN',
                                            'FP/TP', 'Sensitivity', 'Precision'])
pd.set_option('display.width', 200)
print(results_04.round(4))

                            Μοντέλο     AUC  NDCG@1%       TP        FP        FN    FP/TP  Sensitivity  Precision
0              Βάση (καμία τεχνική)  0.7329   0.1396  24.0000  176.0000  143.0000   7.3333       0.1437     0.1200
1              scale_pos_weight = 5  0.7509   0.1135  20.0000  180.0000  147.0000   9.0000       0.1198     0.1000
2        scale_pos_weight, Optuna Α  0.7905   0.1579  26.0000  174.0000  141.0000   6.6923       0.1557     0.1300
3        scale_pos_weight, Optuna Β  0.7942   0.1420  25.0000  175.0000  142.0000   7.0000       0.1497     0.1250
4            RUS 1:4 (μ.ό. 3 seeds)  0.7426   0.1059  17.6667  182.3333  149.3333  10.3208       0.1058     0.0883
5  RUS 1:4, Optuna Α (μ.ό. 3 seeds)  0.7757   0.1440  22.6667  177.3333  144.3333   7.8235       0.1357     0.1133
6      RUS, Optuna Β (μ.ό. 3 seeds)  0.7840   0.1859  24.6667  175.3333  142.3333   7.1081       0.1477     0.1233


In [ ]:
# Feature importance για τις 4 εκδοχές του Optuna: gain (δεδομένα εκπαίδευσης) και permutation (validation)

def fit_model(X, y, params, spw=1):
    # Εκπαίδευση XGBoost με τις δοσμένες ρυθμίσεις -> επιστρέφει το εκπαιδευμένο μοντέλο
    model = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=spw,
                          n_estimators=params['n_estimators'],
                          learning_rate=params['learning_rate'],
                          max_depth=params['max_depth'],
                          min_child_weight=params['min_child_weight'],
                          subsample=params['subsample'],
                          colsample_bytree=params['colsample_bytree'])
    model.fit(X, y)
    return model

def rus_models(k_rus, params):
    # RUS: ένα εκπαιδευμένο μοντέλο για κάθε seed
    models = []
    for s in rus_seeds:
        non_sample = non_frauds.sample(n=len(frauds) * k_rus, random_state=s)
        train_r = pd.concat([frauds, non_sample])
        models.append(fit_model(train_r[raw_scaled], train_r['misstate'], params))
    return models

def gain_importance(model, features):
    gain = model.get_booster().get_score(importance_type='gain')   # μόνο όσες χρησιμοποιήθηκαν
    gain = pd.Series(gain).reindex(features, fill_value=0)         # όσες δεν χρησιμοποιήθηκαν -> 0
    return gain / gain.sum() * 100                                 # ποσοστό (%) επί του συνόλου

# (όνομα, λίστα εκπαιδευμένων μοντέλων): 1 για το scale_pos_weight, 3 για το RUS
models_04_imp = [
    ('scale_pos_weight, Optuna Α', [fit_model(X_all, y_all, best_A_spw, spw=5)]),
    ('scale_pos_weight, Optuna Β', [fit_model(X_all, y_all, best_B_spw, spw=best_B_spw['scale_pos_weight'])]),
    ('RUS 1:4, Optuna Α',          rus_models(4, best_A_rus)),
    ('RUS, Optuna Β',              rus_models(best_B_rus['k_rus'], best_B_rus)),
]

gain_04 = {}
perm_04 = {}
for name, models in models_04_imp:
    gains = []
    reps = []
    for model in models:
        gains.append(gain_importance(model, raw_scaled))
        result = permutation_importance(model, valid[raw_scaled], valid['misstate'],
                                        scoring='roc_auc', n_repeats=10,
                                        random_state=RANDOM_STATE)
        reps.append(result.importances)                  # πίνακας: μεταβλητές × 10 επαναλήψεις

    # gain: μέσος όρος των μοντέλων (1 ή 3)
    gain_04[name] = pd.concat(gains, axis=1).mean(axis=1).sort_values(ascending=False)

    # permutation: όλες οι επαναλήψεις μαζί (10 ή 30)
    all_reps = np.hstack(reps)
    p = pd.DataFrame({'mean': all_reps.mean(axis=1), 'std': all_reps.std(axis=1)}, index=raw_scaled)
    p['abs'] = p['mean'].abs()
    perm_04[name] = p.sort_values('abs', ascending=False)

pd.set_option('display.width', 300)

# Α) Gain: οι 5 πιο σημαντικές μεταβλητές (%)
top5_gain = pd.DataFrame(index=[1, 2, 3, 4, 5])
for name, g in gain_04.items():
    labels = []
    for var, val in g.head(5).items():
        labels.append(var + " (" + str(round(val, 1)) + "%)")
    top5_gain[name] = labels
print("GAIN")
print(top5_gain.T.to_string())
print()

# Β) Permutation: οι 5 με τη μεγαλύτερη επίδραση (θετική ή αρνητική), τιμή ± τυπική απόκλιση
top5_perm = pd.DataFrame(index=[1, 2, 3, 4, 5])
for name, p in perm_04.items():
    labels = []
    for var, row in p.head(5).iterrows():
        labels.append(var + " (" + str(round(row['mean'], 4)) + " ± " + str(round(row['std'], 4)) + ")")
    top5_perm[name] = labels
print("PERMUTATION")
print(top5_perm.T.to_string())
print()

# Γ) Permutation: σαφώς χρήσιμες και σαφώς επιβλαβείς (η επίδραση ξεπερνά τις 2 τυπικές αποκλίσεις)
for name, p in perm_04.items():
    useful  = p[p['mean'] - 2 * p['std'] > 0].index
    harmful = p[p['mean'] + 2 * p['std'] < 0].index
    print(name)
    print("   σαφώς χρήσιμες (", len(useful), "από", len(p), "):", ", ".join(useful))
    print("   σαφώς επιβλαβείς (", len(harmful), "):", ", ".join(harmful))

GAIN
                                          1                2              3              4               5
scale_pos_weight, Optuna Α   sstk_at (4.5%)  ppegt_at (4.4%)  lct_at (4.1%)  che_at (4.0%)    dp_at (3.9%)
scale_pos_weight, Optuna Β   sale_at (4.1%)    lct_at (3.9%)   dp_at (3.8%)   re_at (3.8%)  cogs_at (3.8%)
RUS 1:4, Optuna Α           ppegt_at (5.9%)   sstk_at (5.2%)   dp_at (4.6%)  che_at (4.2%)  rect_at (4.0%)
RUS, Optuna Β                sstk_at (5.7%)  ppegt_at (5.5%)   dp_at (4.6%)  che_at (4.2%)     csho (3.9%)

PERMUTATION
                                                     1                           2                        3                          4                         5
scale_pos_weight, Optuna Α    prcc_f (0.0507 ± 0.0036)  ppegt_at (0.0428 ± 0.0125)  dp_at (0.0263 ± 0.0085)  sstk_at (0.0235 ± 0.0052)   re_at (0.0218 ± 0.0042)
scale_pos_weight, Optuna Β    prcc_f (0.0629 ± 0.0048)  ppegt_at (0.0622 ± 0.0127)   csho (0.0322 ± 0.0045)    dp_at (0.0213 